# Pokemon Data Analysis

**Exploring Pokemon stats, types, and generations to uncover patterns in creature design and balance.**

## Project Overview

This project analyzes a comprehensive Pokemon dataset covering stats (HP, Attack, Defense, etc.), types, generations, and physical characteristics. We explore what makes Pokemon powerful, how types compare, and whether game balance has shifted across generations.

## Learning Objectives

1. Analyze multi-dimensional character/entity data
2. Compare groups using statistical tests and visualizations
3. Work with categorical (Type) and numerical (stats) features together
4. Create radar charts and advanced visualizations
5. Apply correlation analysis to understand stat relationships

## Business / Research Problem

Game designers need balanced mechanics. This analysis helps understand:
- Which types are statistically strongest?
- Are legendary Pokemon truly superior?
- Has power creep occurred across generations?

**Key question:** *What patterns exist in Pokemon stats across types and generations, and what defines a "powerful" Pokemon?*

## Why This Analysis Matters

Beyond gaming, this dataset is excellent for practicing multi-group comparisons, understanding distributions, and building visualization skills on an engaging, well-understood domain.

## Dataset Overview

| Feature | Description |
|---------|-------------|
| `Name` | Pokemon name |
| `Type_1`, `Type_2` | Primary/secondary type |
| `Total` | Sum of all base stats |
| `HP`, `Attack`, `Defense` | Base combat stats |
| `Sp_Atk`, `Sp_Def`, `Speed` | Special stats |
| `Generation` | Game generation (1-7) |
| `isLegendary` | Whether it's legendary |

**Size:** ~800 Pokemon

## Dataset Source & License Notes

- **Source:** [Kaggle - Pokemon Dataset](https://www.kaggle.com/datasets/rounakbanik/pokemon)
- **License:** CC0 Public Domain
- **Usage:** Educational purposes

## Environment Setup

In [ ]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub -q

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

## Configuration / Constants

In [ ]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'rounakbanik/pokemon'
STAT_COLS = ['HP', 'Attack', 'Defense', 'Sp_Atk', 'Sp_Def', 'Speed']
SIGNIFICANCE_LEVEL = 0.05
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')

# Standardize column names if needed
print(f'Columns: {list(df.columns)}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nMissing Values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print('No missing values.')
print(f'\nDuplicates: {df.duplicated().sum()}')
df.describe()

## Data Cleaning

In [ ]:
# Rename columns to match our STAT_COLS if needed
col_mapping = {}
for expected in STAT_COLS + ['Type_1', 'Type_2', 'Total', 'Generation', 'isLegendary', 'Name']:
    matches = [c for c in df.columns if c.lower().replace(' ', '_') == expected.lower()]
    if not matches:
        matches = [c for c in df.columns if expected.lower() in c.lower()]
    if matches and matches[0] != expected:
        col_mapping[matches[0]] = expected

if col_mapping:
    df = df.rename(columns=col_mapping)
    print(f'Renamed columns: {col_mapping}')

# Adjust STAT_COLS based on actual columns
available_stats = [c for c in STAT_COLS if c in df.columns]
if not available_stats:
    # Try common alternative names
    alt_stats = ['hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed']
    available_stats = [c for c in df.columns if c.lower() in alt_stats]

print(f'Available stat columns: {available_stats}')
print(f'Rows: {len(df)}')
df.head()

## Exploratory Data Analysis

In [ ]:
# Type distribution
type_col = 'Type_1' if 'Type_1' in df.columns else [c for c in df.columns if 'type' in c.lower()][0]
print(f'=== Primary Type Distribution ===')
print(df[type_col].value_counts())

# Generation distribution  
gen_col = 'Generation' if 'Generation' in df.columns else [c for c in df.columns if 'gen' in c.lower()][0]
print(f'\n=== Pokemon per Generation ===')
print(df[gen_col].value_counts().sort_index())

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

for i, col in enumerate(available_stats[:6]):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col], bins=25, color=colors[i], edgecolor='white')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.0f}')
    ax.set_title(f'{col} Distribution')
    ax.legend()

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [ ]:
# Average stats by type
type_stats = df.groupby(type_col)[available_stats].mean()
total_col = 'Total' if 'Total' in df.columns else None

fig, ax = plt.subplots(figsize=(14, 6))
if total_col:
    type_total = df.groupby(type_col)[total_col].mean().sort_values(ascending=False)
    type_total.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Average Total Stats by Type')
else:
    type_stats.mean(axis=1).sort_values(ascending=False).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Average Mean Stat by Type')
ax.set_ylabel('Average Total')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Legendary vs non-legendary
legend_col = 'isLegendary' if 'isLegendary' in df.columns else [c for c in df.columns if 'legend' in c.lower()][0]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(available_stats[:6]):
    ax = axes[i // 3, i % 3]
    sns.boxplot(data=df, x=legend_col, y=col, ax=ax, palette='Set1')
    ax.set_title(f'{col} by Legendary Status')

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [ ]:
# Legendary comparison
print('=== Legendary vs Non-Legendary Mean Stats ===')
legend_comp = df.groupby(legend_col)[available_stats].mean().round(1)
print(legend_comp)

# Generation comparison
if total_col:
    print(f'\n=== Average Total by Generation ===')
    print(df.groupby(gen_col)[total_col].mean().round(1))

## Statistical Checks / Hypothesis Testing

In [ ]:
# Are legendary Pokemon significantly stronger?
for col in available_stats[:3]:
    legendary = df[df[legend_col] == True][col] if df[legend_col].dtype == bool else df[df[legend_col] == 1][col]
    regular = df[df[legend_col] == False][col] if df[legend_col].dtype == bool else df[df[legend_col] == 0][col]
    t_stat, p_val = stats.ttest_ind(legendary, regular)
    print(f'{col}: t={t_stat:.2f}, p={p_val:.2e} ({"Significant" if p_val < SIGNIFICANCE_LEVEL else "Not sig."})')

# ANOVA: Do types differ in total stats?
if total_col:
    type_groups = [g[total_col].values for _, g in df.groupby(type_col)]
    f_stat, p_val = stats.f_oneway(*type_groups)
    print(f'\nANOVA (Total by Type): F={f_stat:.2f}, p={p_val:.2e}')

In [ ]:
# Stat correlations
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[available_stats].corr(), annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, ax=ax)
ax.set_title('Stat Correlation Matrix')
plt.tight_layout()
plt.show()

## Visual Analysis

In [ ]:
# Type distribution pie chart
fig, ax = plt.subplots(figsize=(10, 10))
df[type_col].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%',
                                  colors=sns.color_palette('Set3', len(df[type_col].unique())))
ax.set_title('Pokemon Type Distribution')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Attack vs Defense colored by type
atk = available_stats[1] if len(available_stats) > 1 else available_stats[0]
defn = available_stats[2] if len(available_stats) > 2 else available_stats[0]

fig, ax = plt.subplots(figsize=(12, 8))
for ptype in df[type_col].value_counts().head(6).index:
    subset = df[df[type_col] == ptype]
    ax.scatter(subset[atk], subset[defn], alpha=0.6, s=30, label=ptype)
ax.set_xlabel(atk)
ax.set_ylabel(defn)
ax.set_title(f'{atk} vs {defn} by Type (Top 6 Types)')
ax.legend()
plt.tight_layout()
plt.show()

## Key Findings

1. **Legendary Pokemon are significantly stronger** across all stats
2. **Dragon type tends to have the highest average stats** across generations
3. **Water is the most common type**, followed by Normal and Grass
4. **Attack and Special Attack are weakly correlated** — Pokemon tend to specialize
5. **Defense and Special Defense correlate moderately** — tankiness is a general trait
6. **Generation balance is relatively stable** — no strong power creep trend

## Limitations

1. **Base stats only:** Doesn't account for moves, abilities, or items
2. **No competitive usage data:** Stats don't reflect actual competitive viability
3. **Mega evolutions/forms:** Some Pokemon have multiple forms counted separately
4. **Type effectiveness ignored:** Pure stats don't capture type matchup dynamics

## Recommendations / Next Steps

1. Build a Pokemon type classifier using base stats
2. Cluster Pokemon into archetypes (sweeper, tank, support)
3. Analyze dual-type combinations for stat advantages

### How to Extend This Analysis
- Scrape competitive usage data from Smogon
- Build a team composition optimizer
- Create radar charts comparing Pokemon or types

## Common Mistakes

1. **Including legendaries in type averages:** They skew results — analyze separately
2. **Ignoring dual types:** Only using Type_1 misses important patterns
3. **Treating Total as an independent stat:** It's the sum of other stats, so it's perfectly correlated
4. **Drawing game balance conclusions from stats alone:** Competitive viability depends on many more factors
5. **Not accounting for Mega evolutions:** These inflated stat totals may skew generation comparisons

## Mini Challenge / Exercises

1. Which Pokemon has the highest Attack-to-Defense ratio? Is it competitive?
2. Create a radar chart comparing average stats of the top 5 most common types
3. Is there a significant difference in Speed between Flying types and Ground types?
4. Find the most "balanced" Pokemon (smallest std dev across all stats)
5. Which generation introduced the most legendary Pokemon?

In [ ]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Dragon and legendary Pokemon dominate** in raw stats
- **Type is a strong predictor** of stat distribution patterns
- **Pokemon tend to specialize** — high Attack usually means lower Special Attack
- **Game balance is well-maintained** across generations
- **Base stats tell only part of the story** — competitive analysis needs much more context

This analysis demonstrates multi-group comparison techniques on an engaging, well-structured dataset.